# Capability 08 — SFA forecasting

Train naive, moving-average, ARIMA and Facebook Prophet models on a
deterministic daily retailer-demand panel. The notebook compares hold-out
error, exports the winning artifact the API loads, and shows the RET-001
stockout. Forecasts stay derived.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from train import run_training

ROOT = Path(".")
for folder in ("tables", "plots"):
    (ROOT / "outputs" / folder).mkdir(parents=True, exist_ok=True)
(ROOT / "artifacts").mkdir(parents=True, exist_ok=True)
result = run_training(ROOT)
result["metrics"]

/home/ubuntu/.cache/pypoetry/virtualenvs/telco-digital-xS3fZVNL-py3.12/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


/home/ubuntu/.cache/pypoetry/virtualenvs/telco-digital-xS3fZVNL-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


01:22:57 - cmdstanpy - INFO - Chain [1] start processing


01:22:57 - cmdstanpy - INFO - Chain [1] done processing


{'model_version': 'sfa-forecast-v1',
 'served_model': 'arima',
 'prophet_backend': 'prophet',
 'comparison': {'naive': {'mae': 1.1305, 'rmse': 1.5991, 'mape': 25.0031},
  'seasonal_naive': {'mae': 0.7049, 'rmse': 0.792, 'mape': 11.1372},
  'moving_average': {'mae': 1.5537, 'rmse': 1.6281, 'mape': 26.6045},
  'arima': {'mae': 0.0018, 'rmse': 0.0027, 'mape': 0.0367},
  'prophet': {'mae': 0.6662, 'rmse': 0.774, 'mape': 13.1731}},
 'reconstruction': {'mae': 2.9528, 'rmse': 3.6975, 'mape': 43.894},
 'hero': {'retailer_ref': 'RET-001',
  'product_code': 'POC-PROD-01',
  'as_of': '2026-08-21',
  'on_hand': 18.26,
  'forecast_7d': 49.06,
  'actual_7d': 47.34,
  'stockout_warning': True}}

In [2]:
comparison = pd.DataFrame(
    [{"model": name, **values} for name, values in result["metrics"]["comparison"].items()]
).sort_values("mape")
comparison

,model,mae,rmse,mape
3,arima,0.0018,0.0027,0.0367
1,seasonal_naive,0.7049,0.7920,11.1372
4,prophet,0.6662,0.7740,13.1731
0,naive,1.1305,1.5991,25.0031
2,moving_average,1.5537,1.6281,26.6045


In [3]:
holdout = result["holdout"].copy()
holdout["actual"] = holdout["y"]
for name, predicted in result["forecasts"].items():
    holdout[name] = predicted
holdout[["ds", "actual", "naive", "seasonal_naive", "moving_average", "arima", "prophet"]].tail(7)

,ds,actual,naive,seasonal_naive,moving_average,arima,prophet
348,2026-08-15 00:00:00+00:00,4.6814,6.8479,3.8693,5.5081,4.674631,5.783799
349,2026-08-16 00:00:00+00:00,3.9257,6.8479,3.2482,5.5081,3.916928,5.499729
350,2026-08-17 00:00:00+00:00,7.1104,6.8479,5.8898,5.5081,7.108007,6.823247
351,2026-08-18 00:00:00+00:00,7.4196,6.8479,6.1525,5.5081,7.417621,6.971368
352,2026-08-19 00:00:00+00:00,7.7321,6.8479,6.4185,5.5081,7.730640,7.119942
353,2026-08-20 00:00:00+00:00,7.3773,6.8479,6.1304,5.5081,7.375324,6.998924
354,2026-08-21 00:00:00+00:00,8.2322,6.8479,6.8479,5.5081,8.231624,7.365400


In [4]:
result["metrics"]["hero"]

{'retailer_ref': 'RET-001',
 'product_code': 'POC-PROD-01',
 'as_of': '2026-08-21',
 'on_hand': 18.26,
 'forecast_7d': 49.06,
 'actual_7d': 47.34,
 'stockout_warning': True}

In [5]:
fig, axis = plt.subplots(figsize=(7, 3.4))
axis.bar(comparison["model"], comparison["mape"], color=["#1890ff", "#722ed1", "#d48806", "#389e0d", "#d4380d"][: len(comparison)])
axis.set_ylabel("Hold-out MAPE (%)")
axis.set_title("RET-001 / POC-PROD-01 model comparison")
axis.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(ROOT / "outputs" / "plots" / "model_comparison.png", dpi=120)
plt.close(fig)

fig, axis = plt.subplots(figsize=(8, 3.8))
axis.plot(holdout["ds"], holdout["actual"], label="actual", color="#1f1f1f")
axis.plot(holdout["ds"], holdout["arima"], label="ARIMA", color="#1890ff")
axis.plot(holdout["ds"], holdout["prophet"], label="Prophet", color="#722ed1")
axis.plot(holdout["ds"], holdout["seasonal_naive"], label="seasonal naive", color="#8c8c8c", linestyle="--")
axis.set_title("Hold-out daily demand")
axis.legend()
axis.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(ROOT / "outputs" / "plots" / "hero_forecast.png", dpi=120)
plt.close(fig)

hero = result["metrics"]["hero"]
fig, axis = plt.subplots(figsize=(5.4, 3.2))
axis.bar(["On hand", "7-day forecast"], [hero["on_hand"], hero["forecast_7d"]], color=["#d4380d", "#1890ff"])
axis.set_title("RET-001 stockout cover at 2026-08-21")
axis.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(ROOT / "outputs" / "plots" / "stockout_cover.png", dpi=120)
plt.close(fig)
"plots written"

'plots written'